# Train a XGBoost regression model on Amazon SageMaker, host inference on a Docker container running on Amazon ECS on AWS Fargate and optionally expose as an API with Amazon API Gateway

[Amazon SageMaker](https://aws.amazon.com/sagemaker/) is a fully managed end-to-end Machine Learning (ML) service. With SageMaker, you have the option of using the built-in algorithms or you can bring your own algorithms and frameworks to train your models.  After training, you can deploy the models in [one of two ways](https://docs.aws.amazon.com/sagemaker/latest/dg/deploy-model.html) for inference - persistent endpoint or batch transform.

With a persistent inference endpoint, you get a fully-managed real-time HTTPS endpoint hosted on either CPU or GPU based EC2 instances.  It supports features like auto scaling, data capture, model monitoring and also provides cost-effective GPU support using [Amazon Elastic Inference](https://docs.aws.amazon.com/sagemaker/latest/dg/ei.html).  It also supports hosting multiple models using multi-model endpoints that provide A/B testing capability.  You can monitor the endpoint using [Amazon CloudWatch](https://aws.amazon.com/cloudwatch/).  In addition to all these, you can use [Amazon SageMaker Pipelines](https://aws.amazon.com/sagemaker/pipelines/) which provides a purpose-built, easy-to-use Continuous Integration and Continuous Delivery (CI/CD) service for Machine Learning.

There are use cases where you may want to host the ML model on a real-time inference endpoint that is cost-effective and do not require all the capabilities provided by the SageMaker persistent inference endpoint.  These may involve,
* simple models
* models whose sizes are lesser than 200 MB
* models that are invoked sparsely and do not need inference instances running all the time
* models that do not need to be re-trained and re-deployed frequently
* models that do not need GPUs for inference

In these cases, you can take the trained ML model and host it on a container on [Amazon ECS on AWS Fargate](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/AWS_Fargate.html) and optionally expose it as an API by front-ending it with a HTTP/REST API hosted on [Amazon API Gateway](https://aws.amazon.com/api-gateway/).  This will be cost-effective as compared to having real-time inference instances and still provide a fully-managed and scalable solution.

[Amazon Elastic Container Service (Amazon ECS)](https://aws.amazon.com/ecs) is a fully managed container orchestration service.  It is a highly scalable, fast container management service that makes it easy to run, stop and manage containers on a cluster. Your containers are defined in a task definition that you use to run individual tasks or tasks within a service.  In this context, a service is a configuration that enables you to run and maintain a specified number of tasks simultaneously in a cluster. You can run your tasks and services on a serverless infrastructure that is managed by [AWS Fargate](https://aws.amazon.com/fargate). Alternatively, for more control over your infrastructure, you can run your tasks and services on a cluster of Amazon EC2 instances that you manage.

This notebook demonstrates this solution by using SageMaker's [built-in XGBoost algorithm](https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost.html) to train a regression model on the [California Housing dataset](https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html).  It loads the trained model as a Python3 [pickle](https://docs.python.org/3/library/pickle.html) object in a Python3 [Flask](https://flask.palletsprojects.com/en/1.1.x/) app script in a container to be hosted on [Amazon ECS on AWS Fargate](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/AWS_Fargate.html).  Finally, it provides instructions for exposing it as an API by front-ending it with a HTTP/REST API hosted on [Amazon API Gateway](https://aws.amazon.com/api-gateway/).

**Warning:** The Python3 [pickle](https://docs.python.org/3/library/pickle.html) module is not secure.  Only unpickle data you trust.  Keep this in mind if you decide to get the trained ML model file from somewhere instead of building your own model.

**Note:**

* This notebook should be run from a SageMaker Studio JupyterLab Space.
* This notebook uses CPU based instances for training.
* If you already have a trained model that can be loaded as a Python3 [pickle](https://docs.python.org/3/library/pickle.html) object, then you can skip the training step in this notebook and directly upload the model file to S3 and update the code in this notebook's cells accordingly.
* In this notebook, the ML model generated in the training step has not been tuned as that is not the intent of this demo.
* In this notebook, we will create only one ECS Task.  In order to scale to more tasks, you have to create an [Amazon ECS Service](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/ecs_services.html) made up of multiple tasks.  You can then setup [Load Balancing](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/service-load-balancing.html) and [Auto Scaling](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/service-auto-scaling.html).
* This notebook will create resources in the same AWS account and in the same region where this notebook is running.
* Docker is available in your SageMaker environment.  The `docker-buildx` plugin is required since newer Docker versions use BuildKit by default.  The notebook installs it automatically.

**Table of Contents:**

1. [Complete prerequisites](#Complete%20prerequisites)

    1. [Check and configure access to the Internet](#Check%20and%20configure%20access%20to%20the%20Internet)

    2. [Check and upgrade required software versions](#Check%20and%20upgrade%20required%20software%20versions)
    
    3. [Check and configure security permissions](#Check%20and%20configure%20security%20permissions)

    4. [Organize imports](#Organize%20imports)
    
    5. [Create common objects](#Create%20common%20objects)

2. [Prepare the data](#Prepare%20the%20data)

    1. [Create the local directories](#Create%20the%20local%20directories)
    
    2. [Load the dataset and view the details](#Load%20the%20dataset%20and%20view%20the%20details)
    
    3. [(Optional) Visualize the dataset](#(Optional)%20Visualize%20the%20dataset)
    
    4. [Split the dataset into train, validate and test sets](#Split%20the%20dataset%20into%20train,%20validate%20and%20test%20sets)
    
    5. [Standardize the datasets](#Standardize%20the%20datasets)
    
    6. [Save the prepared datasets locally](#Save%20the%20prepared%20datasets%20locally)
    
    7. [Upload the prepared datasets to S3](#Upload%20the%20prepared%20datasets%20to%20S3)

3. [Perform training](#Perform%20training)

    1. [Set the training parameters](#Set%20the%20training%20parameters)
    
    2. [(Optional) Delete previous checkpoints](#(Optional)%20Delete%20previous%20checkpoints)
    
    3. [Run the training job](#Run%20the%20training%20job)

4. [Create and push the Docker container to an Amazon ECR repository](#Create%20and%20push%20the%20Docker%20container%20to%20an%20Amazon%20ECR%20repository)

    1. [Retrieve the model pickle file](#Retrieve%20the%20model%20pickle%20file)
    
    2. [(Optional) Test the model pickle file](#(Optional)%20Test%20the%20model%20pickle%20file)
    
    3. [View the inference script](#View%20the%20inference%20script)
    
    4. [Create the Dockerfile](#Create%20the%20Dockerfile)
    
    5. [Create the container](#Create%20the%20container)
    
    6. [Create the private repository in ECR](#Create%20the%20private%20repository%20in%20ECR)
    
    7. [Push the container to ECR](#Push%20the%20container%20to%20ECR)

5. [Deploy and test on Amazon ECS on AWS Fargate](#Deploy%20and%20test%20on%20Amazon%20ECS%20on%20AWS%20Fargate)
    
    1. [Create the ECS cluster](#Create%20the%20ECS%20cluster)
    
    2. [Create the ECS Task and deploy the container](#Create%20the%20ECS%20Task%20and%20deploy%20the%20container)
    
    3. [Prepare to test the ECS Task](#Prepare%20to%20test%20the%20ECS%20Task)
    
    4. [Test the ECS Task](#Test%20the%20ECS%20Task)
    
6. [(Optional) Front-end the container with Amazon API Gateway](#(Optional)%20Front-end%20the%20container%20with%20Amazon%20API%20Gateway)

7. [Cleanup](#Cleanup)

    1. [Cleanup ECS resources](#Cleanup%20ECS%20resources)
    
    2. [Cleanup ECR repository](#Cleanup%20ECR%20repository)
    
    3. [Cleanup S3 objects](#Cleanup%20S3%20objects)

##  1. Complete prerequisites <a id='Complete%20prerequisites'></a>

Check and complete the prerequisites.

###  A. Check and configure access to the Internet <a id='Check%20and%20configure%20access%20to%20the%20Internet'></a>

This notebook requires outbound access to the Internet to download the required software updates and to make calls to the container hosted as an Amazon ECS Task.  You can either provide direct Internet access (default) or provide Internet access through a VPC.  For more information on this, refer [here](https://docs.aws.amazon.com/sagemaker/latest/dg/appendix-notebook-and-internet-access.html).

### B. Check and upgrade required software versions  <a id='Check%20and%20upgrade%20required%20software%20versions'></a>

This notebook is validated against the CPU image of [Amazon SageMaker Distribution 4.3.3](https://github.com/aws/sagemaker-distribution/blob/main/build_artifacts/v4/v4.3/v4.3.3/RELEASE.md).  Its baseline includes:

* Python 3.12.13
* SageMaker Python SDK 3.12.0 (the notebook uses its SageMaker Core `ModelTrainer` API, not the legacy SDK v2 API)
* Boto3 1.43.46
* NumPy 1.26.4, pandas 2.3.3, scikit-learn 1.7.2, matplotlib 3.10.9, and seaborn 0.13.2
* XGBoost 2.1.4 (CPU image)
* Docker CLI 29.6.2 with a usable Docker daemon and [BuildKit/buildx](https://docs.docker.com/build/buildx/)
* [AWS Command Line Interface](https://aws.amazon.com/cli/) and [cURL](https://curl.se/)

The following cell displays the installed versions.  Newer compatible patch versions are acceptable; the notebook does not upgrade preinstalled packages automatically (it only installs XGBoost if it is absent).

**Note:** When running the following cell, if you get 'module not found' errors, then uncomment the appropriate installation commands and install the modules.  Also, uncomment and run the kernel shutdown command.  When the kernel comes back, comment out the installation and kernel shutdown commands and run the following cell.  Now, you should not see any errors.

In [ ]:
import sys
import subprocess
from importlib.metadata import version as pkg_version

# Ensure xgboost is available
try:
    import xgboost as xgb
except ModuleNotFoundError:
    print('Installing XGBoost module...')
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "xgboost"])
    import xgboost as xgb

import boto3
import os
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core import image_uris
from sagemaker.train import ModelTrainer

# Display the installed runtime against the SageMaker Distribution 4.3.3 baseline.
SMD_433_BASELINE = {
    'numpy': '1.26.4',
    'pandas': '2.3.3',
    'scikit-learn': '1.7.2',
    'matplotlib': '3.10.9',
    'seaborn': '0.13.2',
    'boto3': '1.43.46',
    'xgboost': '2.1.4',
}

print('SageMaker Distribution baseline: 4.3.3')
print(f"Python version: {sys.version}")
if sys.version_info[:2] != (3, 12):
    print('WARNING: SageMaker Distribution 4.3.3 uses Python 3.12.13; verify package compatibility.')

print(f"SageMaker SDK components: sagemaker-core {pkg_version('sagemaker-core')}, "
      f"sagemaker-train {pkg_version('sagemaker-train')}")
for package_name, baseline_version in SMD_433_BASELINE.items():
    try:
        installed_version = xgb.__version__ if package_name == 'xgboost' else pkg_version(package_name)
        print(f"{package_name}: {installed_version} (4.3.3 baseline: {baseline_version})")
    except Exception as error:
        print(f"{package_name}: not available ({error})")

# Get the AWS CLI version
print("AWS CLI version:")
!aws --version

**Docker:**

SageMaker Distribution 4.3.3 includes Docker CLI 29.6.2.  Building and pushing the inference image also requires a usable Docker daemon.  The following cell verifies Docker and Buildx, then installs Buildx 0.36.1 only if it is missing; it does not overwrite a plugin already provided by the environment.

In [ ]:
import platform
import shutil
import subprocess
from pathlib import Path


def run_checked(command, **kwargs):
    """Run a command and raise a visible error when it fails."""
    print(f'$ {" ".join(command)}')
    return subprocess.run(command, check=True, text=True, **kwargs)


# Verify that the Docker CLI and daemon are usable before spending time building.
if shutil.which('docker') is None:
    raise RuntimeError('Docker CLI was not found. Select a SageMaker environment with Docker support.')

run_checked(['docker', '--version'])
try:
    run_checked(['docker', 'info'], stdout=subprocess.DEVNULL)
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        'Docker CLI is installed but the Docker daemon is unavailable. '
        'Start a SageMaker environment with Docker support before continuing.'
    ) from error

# SageMaker Distribution 4.3.3 includes a modern Docker CLI. Reuse its Buildx
# plugin when available; download a known-compatible plugin only as a fallback.
buildx_check = subprocess.run(
    ['docker', 'buildx', 'version'], text=True, capture_output=True
)
if buildx_check.returncode == 0:
    print(f'Using installed Buildx: {buildx_check.stdout.strip()}')
else:
    architecture = platform.machine().lower()
    buildx_architecture = {
        'x86_64': 'amd64',
        'amd64': 'amd64',
        'aarch64': 'arm64',
        'arm64': 'arm64',
    }.get(architecture)
    if buildx_architecture is None:
        raise RuntimeError(f'Unsupported architecture for Buildx download: {architecture}')

    buildx_version = 'v0.36.1'
    plugin_dir = Path.home() / '.docker' / 'cli-plugins'
    plugin_dir.mkdir(parents=True, exist_ok=True)
    plugin_path = plugin_dir / 'docker-buildx'
    buildx_url = (
        f'https://github.com/docker/buildx/releases/download/{buildx_version}/'
        f'buildx-{buildx_version}.linux-{buildx_architecture}'
    )
    run_checked(['curl', '--fail', '--location', '--silent', '--show-error',
                 buildx_url, '--output', str(plugin_path)])
    plugin_path.chmod(0o755)
    print(f'Installed Buildx {buildx_version} at {plugin_path}')

run_checked(['docker', 'buildx', 'version'])

**cURL:**

cURL should be pre-installed in your SageMaker environment.  Verify it by running the `curl --version` command.

In [ ]:
# Verify if cURL is installed
!curl --version

# Install cURL (uncomment if needed):
#!sudo apt-get update && sudo apt-get install -y curl

**Amazon ECR credential helper:**

Install and configure the [Amazon ECR credential helper](https://github.com/awslabs/amazon-ecr-credential-helper) so that `docker push` can authenticate to your private ECR registry automatically.

In [ ]:
# Install and configure the Amazon ECR credential helper
!sudo apt-get update -qq && sudo apt-get install -y -qq amazon-ecr-credential-helper

# Verify installation
print('\nAmazon ECR Docker Credential Helper version:')
!docker-credential-ecr-login version

# Create the .docker directory if it doesn't exist
!mkdir -p ~/.docker

# Configure Docker to use the ECR credential helper
!printf '{\n\t"credsStore": "ecr-login"\n}' > ~/.docker/config.json

# Verify configuration
print('\nDocker config:')
!cat ~/.docker/config.json

###  C. Check and configure security permissions <a id='Check%20and%20configure%20security%20permissions'></a>

This notebook uses the IAM execution role attached to the underlying SageMaker environment.  This role should have the following permissions,

1. Full access to the S3 bucket that will be used to store training and output data.
2. Full access to launch training instances.
3. Access to create CloudWatch Log Groups.
4. Access to write to CloudWatch Logs and CloudWatch Metrics.
5. Access to create, delete and write to Amazon ECR private registries.
6. Access to create and delete Amazon ECS clusters and a task definitions.
7. Access to run ECS tasks.

To view the name of this role, run the following cell.

In [ ]:
# get_execution_role was imported in Cell 7
print(get_execution_role())

This notebook creates a [task](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/task_definitions.html) on [Amazon ECS on AWS Fargate](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/AWS_Fargate.html).  This task requires an IAM role named 'Task Execution IAM role' that it assumes when it is invoked.  For more information on this, refer [here](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/task_execution_IAM_role.html).

For the task created in this notebook, at a minimum, this role should provide access to the following,

* Access to create CloudWatch Log Groups.
* Access to write to CloudWatch Logs and CloudWatch Metrics.
* Read access to Amazon ECR.

For information on the various IAM roles required for Amazon ECS on AWS Fargate refer [here](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/task-iam-roles.html).

###  D. Organize imports <a id='Organize%20imports'></a>

Organize all the library and module imports for later use.

In [3]:
from io import StringIO
import json
import logging
import matplotlib.pyplot as plt
import numpy as np
import pickle
import pandas as pd
import seaborn as sns
import sklearn.model_selection
from sklearn.preprocessing import StandardScaler
import tarfile
import time

# Note: Session, get_execution_role, image_uris, and ModelTrainer
# were imported in the prerequisites cell above.

###  E. Create common objects <a id='Create%20common%20objects'></a>

Create common objects to be used in future steps in this notebook.

This notebook reads the S3 bucket, IAM role ARNs, subnet, and security group directly from the outputs of the prerequisites CloudFormation stack (see `deployment/notebook-prerequisites.yaml` and `deployment/setup-prerequisites.sh`).  This removes the need to copy-paste those values by hand.

First, run the cell below to load the stack outputs.  Set `PREREQ_STACK_NAME` to match the stack you deployed (default: `ml-notebook-prereqs`).  If you did not use the CloudFormation template, set `USE_CFN_OUTPUTS = False` and fill in the values manually in the next cell.

In [ ]:
# Load prerequisite values from the CloudFormation stack outputs.
# These map to the Outputs defined in deployment/notebook-prerequisites.yaml.

# Name of the deployed prerequisites stack.
PREREQ_STACK_NAME = 'ml-notebook-prereqs'

# Set to False to skip CloudFormation and configure the values manually below.
USE_CFN_OUTPUTS = True

# Will hold the resolved values; populated from the stack outputs when enabled.
prereq = {}

if USE_CFN_OUTPUTS:
    from botocore.exceptions import ClientError

    cfn_client = boto3.client('cloudformation')
    try:
        stacks = cfn_client.describe_stacks(StackName=PREREQ_STACK_NAME)['Stacks']
    except ClientError as error:
        raise RuntimeError(
            f"Could not find CloudFormation stack '{PREREQ_STACK_NAME}'. "
            f"Deploy it first with deployment/setup-prerequisites.sh, or set "
            f"USE_CFN_OUTPUTS = False and fill in the values manually."
        ) from error

    # Map CloudFormation output keys -> friendly names used in this notebook.
    outputs = {o['OutputKey']: o['OutputValue'] for o in stacks[0].get('Outputs', [])}
    required_keys = ['S3BucketName', 'ECSTaskExecutionRoleArn', 'ECSTaskRoleArn',
                     'PublicSubnetId', 'SecurityGroupId']
    missing = [k for k in required_keys if k not in outputs]
    if missing:
        raise RuntimeError(
            f"Stack '{PREREQ_STACK_NAME}' is missing expected outputs: {missing}. "
            f"Make sure it was deployed from deployment/notebook-prerequisites.yaml."
        )
    prereq = outputs

    print(f"Loaded prerequisite values from stack '{PREREQ_STACK_NAME}':")
    for key in required_keys:
        print(f"  {key} = {prereq[key]}")
else:
    print('USE_CFN_OUTPUTS is False - configure the values manually in the next cell.')

In [ ]:
# S3 bucket name (from the CloudFormation stack output, or set manually if
# USE_CFN_OUTPUTS was False in the previous cell).
s3_bucket = prereq.get('S3BucketName', '<Specify the S3 bucket name>')

# Create the S3 Boto3 resource
s3_resource = boto3.resource('s3')
s3_bucket_resource = s3_resource.Bucket(s3_bucket)

# Create the SageMaker Boto3 client
sm_client = boto3.client('sagemaker')

# Create the ECR client
ecr_client = boto3.client('ecr')

# Create the Amazon ECS client
ecs_client = boto3.client('ecs')

# Create the Amazon EC2 client
ec2_client = boto3.client('ec2')

# Create SageMaker session and get the AWS region name
sagemaker_session = Session()
region_name = sagemaker_session.boto_region_name

# Base name to be used to create resources
nb_name = 'sm-xgboost-ca-housing-ecs-container-model-hosting'

# Names of various resources
train_job_name = f'train-{nb_name}'

# Names of local sub-directories in the notebook file system
data_dir = os.path.join(os.getcwd(), f'data/{nb_name}')
train_dir = os.path.join(os.getcwd(), f'data/{nb_name}/train')
val_dir = os.path.join(os.getcwd(), f'data/{nb_name}/validate')
test_dir = os.path.join(os.getcwd(), f'data/{nb_name}/test')

# Location of the datasets file in the notebook file system
dataset_csv_file = os.path.join(os.getcwd(), 'datasets/california_housing.csv')

# Container artifacts directory in the notebook file system
container_artifacts_dir = os.path.join(os.getcwd(), f'container-artifacts/{nb_name}')

# Location of the Python3 Flask script (containing the inference code) and it's corresponding
# requirements.txt in the notebook file system
container_script_file_name = 'container_sm_xgboost_ca_housing_inference.py'
container_script_req_file_name = 'container_sm_xgboost_ca_housing_inference_requirements.txt'
container_script_file = os.path.join(os.getcwd(), f'scripts/{container_script_file_name}')
container_script_req_file = os.path.join(os.getcwd(), f'scripts/{container_script_req_file_name}')

# Sub-folder names in S3
train_dir_s3_prefix = f'{nb_name}/data/train'
val_dir_s3_prefix = f'{nb_name}/data/validate'
test_dir_s3_prefix = f'{nb_name}/data/test'

# Location in S3 where the model checkpoint will be stored
model_checkpoint_s3_path = f's3://{s3_bucket}/{nb_name}/checkpoint/'

# Location in S3 where the trained model will be stored
model_output_s3_path = f's3://{s3_bucket}/{nb_name}/output/'

# Names of the model tar file and extracted file - these are dependent on the
# framework and algorithm you used to train the model.  This notebook uses
# SageMaker's built-in XGBoost algorithm and that will have the names as follows:
model_tar_file_name = 'model.tar.gz'
extracted_model_file_name = 'xgboost-model'

# Container details
container_image_name = nb_name
# ECR registry URL prefix is derived from the account ID and region
# (format: {aws_account_id}.dkr.ecr.{region}.amazonaws.com).
aws_account_id = boto3.client('sts').get_caller_identity()['Account']
container_registry_url_prefix = f'{aws_account_id}.dkr.ecr.{region_name}.amazonaws.com'

# ECS cluster details
ecs_cluster_name = f'cluster-{nb_name}'

# ECS Task details
ecs_fargate_task_name = f'fargate-task-{nb_name}'
ecs_fargate_task_role = prereq.get('ECSTaskRoleArn', '<Specify the ARN for the ECS Task IAM role>')
ecs_fargate_task_execution_role = prereq.get('ECSTaskExecutionRoleArn', '<Specify the ARN for the ECS Task execution IAM role>')
# Fargate CPU/memory must be strings of CPU units and MiB (not '0.25 vCPU' / '0.5 GB').
# Valid pairs: cpu '256' -> memory '512'/'1024'/'2048'; see the Fargate task size table.
ecs_fargate_task_cpu = '256'      # 0.25 vCPU
ecs_fargate_task_memory = '512'   # 0.5 GB
ecs_fargate_task_count = 1

# ECS Task networking details
ecs_fargate_task_subnet_list = [prereq.get('PublicSubnetId', '<Specify the ID of a public subnet in your preferred VPC>')]
ecs_fargate_task_security_group_list = [prereq.get('SecurityGroupId', '<Specify the ID of your Security Group in your preferred VPC>')]

# ECS Task container details
ecs_container_name = f'container-{nb_name}'
ecs_container_port = 80
ecs_container_host_port = 80
ecs_container_healthcheck_command_list = ["CMD-SHELL", "curl -f http://localhost:80/healthcheck || exit 1"]
ecs_container_healthcheck_interval_in_seconds = 30
ecs_container_healthcheck_timeout_in_seconds = 30

## 2. Prepare the data <a id='Prepare%20the%20data'></a>

The [California Housing dataset](https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html) consists of 20,640 observations on housing prices with 9 economic covariates.  These covariates are,

* MedianHouseValue
* MedianIncome
* HousingMedianAge
* TotalRooms
* TotalBedrooms
* Population
* Households
* Latitude
* Longitude

This dataset has been downloaded to the local `datasets` directory and modified as a CSV file with the feature names in the first row.  This will be used in this notebook.

The following steps will help with preparing the datasets for training, validation and testing.

### A) Create the local directories <a id='Create%20the%20local%20directories'></a>

Create the directories in the local system where the dataset will be copied to and processed.

In [ ]:
# Create the local directories if they don't exist
os.makedirs(data_dir, exist_ok=True)
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

### B) Load the dataset and view the details <a id='Load%20the%20dataset%20and%20view%20the%20details'></a>

Check if the CSV file exists in the `datasets` directory and load it into a Pandas DataFrame.  Finally, print the details of the dataset.

In [ ]:
# Check if the dataset file exists and proceed
if os.path.exists(dataset_csv_file):
    print('Dataset CSV file \'{}\' exists.'.format(dataset_csv_file))
    # Load the data into a Pandas DataFrame
    pd_data_frame = pd.read_csv(dataset_csv_file)
    # Print the first 5 records
    #print(pd_data_frame.head(5))
    # Describe the dataset
    print(pd_data_frame.describe())
else:
    print('Dataset CSV file \'{}\' does not exist.'.format(dataset_csv_file))

### C) (Optional) Visualize the dataset <a id='(Optional)%20Visualize%20the%20dataset'></a>

Display the distributions in the dataset.

In [ ]:
# Print the correlation matrix
# numeric_only=True is required since pandas 2.0 (corr() no longer silently
# drops non-numeric columns).
plt.figure(figsize=(11, 7))
sns.heatmap(cbar=False, annot=True, data=(pd_data_frame.corr(numeric_only=True) * 100), cmap='coolwarm')
plt.title('% Correlation Matrix')
plt.show()

### D) Split the dataset into train, validate and test sets <a id='Split%20the%20dataset%20into%20train,%20validate%20and%20test%20sets'></a>

Split the dataset into train, validate and test sets after shuffling.  Split further into x and y sets.

In [ ]:
# Split into train and test datasets after shuffling
train, test = sklearn.model_selection.train_test_split(pd_data_frame, test_size=0.2,
                                                       random_state=35, shuffle=True)
# Split the train dataset further into train and validation datasets after shuffling
train, val = sklearn.model_selection.train_test_split(train, test_size=0.1,
                                                      random_state=25, shuffle=True)

# Define functions to get x and y columns
def get_x(df):
    return df[['median_income','housing_median_age','total_rooms','total_bedrooms',
                 'population','households','latitude','longitude']]
def get_y(df):
    return df[['median_house_value']]

# Load the x and y columns for train, validation and test datasets
x_train = get_x(train)
y_train = get_y(train)
x_val = get_x(val)
y_val = get_y(val)
x_test = get_x(test)
y_test = get_y(test)

# Summarize the datasets
print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print("x_val shape:", x_val.shape)
print("y_val shape:", y_val.shape)
print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)

### E) Standardize the datasets <a id='Standardize%20the%20datasets'></a>

* Standardize the x columns of the train dataset using the `fit_transform()` function of `StandardScaler`.
* Standardize the x columns of the validate and test datasets using the `transform()` function of `StandardScaler`.

In [ ]:
# Standardize the dataset
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)
x_test = scaler.transform(x_test)

### F) Save the prepared datasets locally <a id='Save%20the%20prepared%20datasets%20locally'></a>

Save the prepared train, validate and test datasets to local directories.  Prior to saving, concatenate x and y columns as needed.  Create the directories if they don't exist.

In [ ]:
# Save the prepared dataset (in numpy format) to the local directories as csv files

np.savetxt(os.path.join(train_dir, 'train.csv'),
           np.concatenate((y_train.to_numpy(), x_train), axis=1), delimiter=',')
np.savetxt(os.path.join(train_dir, 'train_x.csv'), x_train)
np.savetxt(os.path.join(train_dir, 'train_y.csv'), y_train.to_numpy())

np.savetxt(os.path.join(val_dir, 'validate.csv'),
           np.concatenate((y_val.to_numpy(), x_val), axis=1), delimiter=',')
np.savetxt(os.path.join(val_dir, 'validate_x.csv'), x_val)
np.savetxt(os.path.join(val_dir, 'validate_y.csv'), y_val.to_numpy())

np.savetxt(os.path.join(test_dir, 'test.csv'),
           np.concatenate((y_test.to_numpy(), x_test), axis=1), delimiter=',')
np.savetxt(os.path.join(test_dir, 'test_x.csv'), x_test)
np.savetxt(os.path.join(test_dir, 'test_y.csv'), y_test.to_numpy())

### G) Upload the prepared datasets to S3 <a id='Upload%20the%20prepared%20datasets%20to%20S3'></a>

Upload the datasets from the local directories to appropriate sub-directories in the specified S3 bucket.

In [ ]:
# Upload the data to S3
train_dir_s3_path = sagemaker_session.upload_data(
    path=f'./data/{nb_name}/train/', bucket=s3_bucket, key_prefix=train_dir_s3_prefix)
val_dir_s3_path = sagemaker_session.upload_data(
    path=f'./data/{nb_name}/validate/', bucket=s3_bucket, key_prefix=val_dir_s3_prefix)
test_dir_s3_path = sagemaker_session.upload_data(
    path=f'./data/{nb_name}/test/', bucket=s3_bucket, key_prefix=test_dir_s3_prefix)

# Capture the S3 locations of the uploaded datasets
train_s3_path = f'{train_dir_s3_path}/train.csv'
train_x_s3_path = f'{train_dir_s3_path}/train_x.csv'
train_y_s3_path = f'{train_dir_s3_path}/train_y.csv'
val_s3_path = f'{val_dir_s3_path}/validate.csv'
val_x_s3_path = f'{val_dir_s3_path}/validate_x.csv'
val_y_s3_path = f'{val_dir_s3_path}/validate_y.csv'
test_s3_path = f'{test_dir_s3_path}/test.csv'
test_x_s3_path = f'{test_dir_s3_path}/test_x.csv'
test_y_s3_path = f'{test_dir_s3_path}/test_y.csv'

##  3. Perform training <a id='Perform%20training'></a>

In this step, SageMaker's [built-in XGBoost algorithm](https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost.html) is used to train a regression model on the [California Housing dataset](https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html).

Note: This model has not been tuned as that is not the intent of this demo.

### A) Set the training parameters <a id='Set%20the%20training%20parameters'></a>

1. Inputs - S3 location of the training and validation data.
2. Hyperparameters.
3. Training instance details:

    1. Instance count
    
    2. Instance type
    
    3. The max run time of the training job
    
    4. (Optional) Use Spot instances.  For more info, refer [here](https://docs.aws.amazon.com/sagemaker/latest/dg/model-managed-spot-training.html).
    
    5. (Optional) The max wait for Spot instances, if using Spot.  This should be larger than the max run time.
    
4. Base job name
5. Appropriate local and S3 directories that will be used by the training job.

In [ ]:
# Set the hyperparameters
hyperparameters = {
    'objective': 'reg:squarederror',
    'max_depth': '6',
    'eta': '0.3',
    'alpha': '3',
    'colsample_bytree': '0.7',
    'num_round': '100'
}

# Set the instance count, instance type, and other parameters
train_instance_count = 1
train_instance_type = 'ml.m5.xlarge'
train_instance_volume_size_in_gb = 5
use_spot_instances = False
spot_max_wait_time_in_seconds = None
max_run_time_in_seconds = 3600
algorithm_name = 'xgboost'
algorithm_version = '1.7-1'
py_version = 'py3'

# Get the container image URI for the specified parameters
container_image_uri = image_uris.retrieve(
    framework=algorithm_name,
    region=region_name,
    version=algorithm_version,
    py_version=py_version,
    instance_type=train_instance_type,
    image_scope='training'
)
print(f"Training image URI: {container_image_uri}")

# Set the training container related parameters
container_log_level = logging.INFO

# Location where the model checkpoints will be stored locally in the container
model_checkpoint_local_dir = '/opt/ml/checkpoints/'

# Location where the trained model will be stored locally in the container
model_local_dir = '/opt/ml/model'

### GPU-Accelerated Training (Optional) <a id='GPU-Accelerated%20Training'></a>

XGBoost supports **GPU-accelerated training** via CUDA, which can provide significant speedups (2–10x) for large datasets and deep trees. SageMaker's built-in XGBoost algorithm (version 1.2+) supports GPU training natively.

**When to use GPU training:**
- Dataset has > 100,000 rows
- `max_depth` > 8
- `num_round` > 500
- You need faster iteration during hyperparameter tuning

**When GPU is NOT beneficial:**
- Small datasets (< 50K rows) — GPU overhead exceeds gains
- Shallow trees (`max_depth` ≤ 4)
- **Inference** — XGBoost inference is tree traversal, not matrix math (CPU is optimal)
- **SARIMA/ARIMA models** — purely CPU-based statistical computation

**To enable GPU training**, change the following parameters in the cell above:

| Parameter | CPU (current) | GPU |
|-----------|:-------------:|:---:|
| `train_instance_type` | `ml.m5.xlarge` | `ml.g4dn.xlarge` or `ml.p3.2xlarge` |
| `tree_method` hyperparameter | (default: `auto`) | `gpu_hist` |

**Cost comparison (us-east-1, on-demand):**

| Instance | GPU | vCPU | RAM | Cost/hr | Typical XGBoost 100K rows |
|----------|-----|------|-----|---------|---------------------------|
| ml.m5.xlarge | None | 4 | 16 GB | ~$0.23 | ~5 min |
| ml.g4dn.xlarge | 1x T4 | 4 | 16 GB | ~$0.74 | ~1.5 min |
| ml.p3.2xlarge | 1x V100 | 8 | 61 GB | ~$3.83 | ~45 sec |

For our California Housing dataset (~20K rows, 100 rounds, max_depth=6), GPU provides marginal benefit — training already completes in ~2 minutes on CPU. GPU training is recommended for production workloads with larger datasets.

Run the following cell to see the GPU-enabled configuration (does not execute training — it only overrides the parameters if you choose to use GPU).

In [ ]:
# ============================================================
# GPU-ACCELERATED TRAINING CONFIGURATION
# ============================================================
# Uncomment the following block to switch to GPU training.
# This overrides the CPU parameters set in the cell above.
#
# NOTE: GPU instances cost more per hour but train faster.
# For this small dataset (~20K rows), the total cost difference
# is minimal (~$0.01 vs ~$0.02 per training run).
# ============================================================

USE_GPU_TRAINING = False  # Set to True to enable GPU training

if USE_GPU_TRAINING:
    # Switch to a GPU instance
    train_instance_type = 'ml.g4dn.xlarge'  # NVIDIA T4 GPU, cost-effective
    # Alternative: 'ml.p3.2xlarge'  # NVIDIA V100, faster but more expensive
    
    # Add GPU-specific hyperparameters
    hyperparameters['tree_method'] = 'gpu_hist'  # GPU-accelerated histogram method
    # hyperparameters['gpu_id'] = '0'           # Optional: specify GPU device
    # hyperparameters['max_bin'] = '256'         # Optional: histogram bins (default 256)
    
    # Re-fetch the container image URI for the GPU instance type
    container_image_uri = image_uris.retrieve(
        framework=algorithm_name,
        region=region_name,
        version=algorithm_version,
        py_version=py_version,
        instance_type=train_instance_type,
        image_scope='training'
    )
    
    print(f'GPU Training enabled!')
    print(f'  Instance type: {train_instance_type}')
    print(f'  tree_method: {hyperparameters["tree_method"]}')
    print(f'  Training image URI: {container_image_uri}')
    print(f'')
    print(f'  XGBoost GPU training uses the gpu_hist tree method which:')
    print(f'    - Builds histograms on GPU memory (faster split finding)')
    print(f'    - Uses CUDA kernels for parallel tree construction')
    print(f'    - Automatically falls back to CPU for operations that')
    print(f'      cannot be parallelized on GPU')
else:
    print(f'CPU Training (default) — instance type: {train_instance_type}')
    print(f'Set USE_GPU_TRAINING = True above to enable GPU acceleration.')

### B) (Optional) Delete previous checkpoints <a id='(Optional)%20Delete%20previous%20checkpoints'></a>

If model checkpoints from previous trainings are found in the S3 checkpoint location specified in the previous step, then training will resume from those checkpoints.  In order to start a fresh training, run the following code cell to delete all checkpoint objects from S3.

In [ ]:
# Delete the checkpoints if you want to train from the beginning; else ignore this code cell
for checkpoint_file in s3_bucket_resource.objects.filter(Prefix='{}/checkpoint/'.format(nb_name)):
    checkpoint_file_key = checkpoint_file.key
    print('Deleting {} ...'.format(checkpoint_file_key))
    s3_resource.Object(s3_bucket_resource.name, checkpoint_file_key).delete()

### C) Run the training job <a id='Run%20the%20training%20job'></a>

Prepare the `estimator` and call the `fit()` method.  This will pull the container containing the specified version of the algorithm in the AWS region and run the training job in the specified type of EC2 instance(s).  The training data will be pulled from the specified location in S3 and training results and checkpoints will be written to the specified locations in S3.

Note: SageMaker Debugger is disabled.

In [ ]:
from sagemaker.core.training.configs import Compute, InputData
from sagemaker.core.shapes.shapes import StoppingCondition, OutputDataConfig, CheckpointConfig

# Configure compute resources
compute = Compute(
    instance_type=train_instance_type,
    instance_count=train_instance_count,
    volume_size_in_gb=train_instance_volume_size_in_gb,
    enable_managed_spot_training=use_spot_instances
)

# Configure stopping condition
stopping_condition = StoppingCondition(
    max_runtime_in_seconds=max_run_time_in_seconds,
    max_wait_time_in_seconds=spot_max_wait_time_in_seconds
)

# Configure output
output_data_config = OutputDataConfig(s3_output_path=model_output_s3_path)

# Configure checkpoints
checkpoint_config = CheckpointConfig(
    s3_uri=model_checkpoint_s3_path,
    local_path=model_checkpoint_local_dir
)

# Configure input data channels
train_data = InputData(channel_name="train", data_source=train_s3_path, content_type="text/csv")
val_data = InputData(channel_name="validation", data_source=val_s3_path, content_type="text/csv")

# Create the ModelTrainer
model_trainer = ModelTrainer(
    training_image=container_image_uri,
    hyperparameters=hyperparameters,
    compute=compute,
    stopping_condition=stopping_condition,
    output_data_config=output_data_config,
    checkpoint_config=checkpoint_config,
    role=get_execution_role(),
    base_job_name=train_job_name,
    sagemaker_session=sagemaker_session
)

# Perform the training
model_trainer.train(input_data_config=[train_data, val_data], wait=True)

print(f"\nTraining job name: {model_trainer._latest_training_job.name}")

##  4. Create and push the Docker container to an Amazon ECR repository <a id='Create%20and%20push%20the%20Docker%20container%20to%20an%20Amazon%20ECR%20repository'></a>

In this step, we will create a Docker container containing the generated model along with its dependencies.  If you bring a pre-trained model, you can upload it to S3 and use it to build the container.  The following steps contains instructions for doing so.

### A) Retrieve the model pickle file <a id='Retrieve%20the%20model%20pickle%20file'></a>

* The model file generated using SageMaker's [built-in XGBoost algorithm](https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost.html) will be a Python pickle file zipped up in a tar file named `model.tar.gz`.  The S3 URI for this file will be available in the `model_data` attribute of the `estimator` object created in the training step.

* If you bring your pre-trained model, you have to specify the S3 URI appropriately in the following cell.

* The zip file needs to be downloaded from S3 and extracted.

* The name of the extracted pickle file will depend on the framework and algorithm that was used to train the model.  In this notebook example, we have used SageMaker's [built-in XGBoost algorithm](https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost.html) and so the pickle file will be named `xgboost-model`.  You will see this when the model tar file is extracted.

In [ ]:
# Create the container artifacts directory if it doesn't exist
os.makedirs(container_artifacts_dir, exist_ok=True)

# Set the file paths
training_job_name = model_trainer._latest_training_job.name
model_tar_file_s3_path_suffix = f'{nb_name}/output/{training_job_name}/output/{model_tar_file_name}'
model_tar_file_local_path = f'{container_artifacts_dir}/{model_tar_file_name}'
extracted_model_file_local_path = f'{container_artifacts_dir}/{extracted_model_file_name}'

# Delete old model files if they exist
if os.path.exists(model_tar_file_local_path):
    os.remove(model_tar_file_local_path)
if os.path.exists(extracted_model_file_local_path):
    os.remove(extracted_model_file_local_path)

# Download the model tar file from S3
s3_bucket_resource.download_file(model_tar_file_s3_path_suffix, model_tar_file_local_path)

# Extract the model tar file and retrieve the model pickle file.
# filter='data' applies the safe extraction filter (Python 3.12+); it avoids a
# DeprecationWarning now and the behavior change coming in Python 3.14.
with tarfile.open(model_tar_file_local_path, "r:gz") as tar:
    tar.extractall(path=container_artifacts_dir, filter='data')

### H) Stage the trained model for the inference app (ECS worker) <a id='Stage%20the%20trained%20model'></a>
Copy the extracted model pickle into the containerized inference application at `app/backend/ecs_worker/models/` using the file name the ECS worker expects (`xgboost-model.pkl`). This automates step 3.1 of `instructions.md` so you do not have to download, extract and rename the artifact by hand before building the worker image. The staged file is git-ignored.


In [ ]:
# Stage the extracted model for the containerized inference app (app/backend/ecs_worker).
# The ECS worker loads '<model-name>-model.pkl' from /opt/models, which is baked into
# the image at build time from app/backend/ecs_worker/models/. This replaces the manual
# fetch/extract/rename described in instructions.md (step 3.1).
import shutil

# Resolve the app's model staging directory relative to this notebook.
# Override ECS_WORKER_MODELS_DIR if your repository layout differs.
ecs_worker_models_dir = os.environ.get(
    'ECS_WORKER_MODELS_DIR',
    os.path.join(os.getcwd(), '..', 'app', 'backend', 'ecs_worker', 'models'),
)
ecs_worker_models_dir = os.path.abspath(ecs_worker_models_dir)
os.makedirs(ecs_worker_models_dir, exist_ok=True)

# SageMaker's built-in XGBoost algorithm produces a pickle named 'xgboost-model';
# the worker expects it as 'xgboost-model.pkl'.
staged_model_path = os.path.join(ecs_worker_models_dir, 'xgboost-model.pkl')
shutil.copyfile(extracted_model_file_local_path, staged_model_path)

staged_size_mb = os.path.getsize(staged_model_path) / (1024 * 1024)
print('Staged trained model for the ECS worker image:')
print(f'  source: {extracted_model_file_local_path}')
print(f'  target: {staged_model_path} ({staged_size_mb:.2f} MB)')
print('\nThis file is git-ignored. Next: build and push the worker image '
      '(instructions.md step 3.2).')


### B) (Optional) Test the model pickle file <a id='(Optional)%20Test%20the%20model%20pickle%20file'></a>

The code in the following cell entirely depends on the framework and algorithm that was used to train the model.  The extracted Python3 pickle file will contain the appropriate object name.  If you are bringing your own model file, you have to change this cell appropriately.

In [ ]:
# Load the model pickle file as a pickle object
pickle_file_path = extracted_model_file_local_path
with open(pickle_file_path, 'rb') as pkl_file:
    model = pickle.load(pkl_file)

# Run a prediction against the model loaded as a pickle object
# by sending the first record of the test dataset
test_pred_x_df = pd.read_csv(StringIO(','.join(map(str, x_test[0]))), sep=',', header=None)
test_pred_x = xgb.DMatrix(test_pred_x_df.values)
print('Input for prediction = {}'.format(test_pred_x_df.values))
print('Predicted value = {}'.format(model.predict(test_pred_x)[0]))
print('Actual value = {}'.format(y_test.values[0][0]))
print('Note: There may be a huge difference between the actual and predicted values as the model has not been tuned in the training step.')

### C) View the inference script <a id='View%20the%20inference%20script'></a>

The inference script is a Python3 [Flask](https://flask.palletsprojects.com/en/1.1.x/) app script that contains the following logic:
* Initialize the Flask web app server.
* Load the ML model pickle object into memory.
* Run the Flask web app server.
* Parse the request sent to the web app server either from direct invocation or from a REST/HTTP API in Amazon API Gateway.
* Run the prediction.
* Format the response to match with the parameter specified in the request.
* Return the response.
* Implement the healthcheck logic to return a success on invocation.  This has to be called by the service hosting this container to perform health checks.

The request should be in the following format:

`{
  "response_content_type": "<Specify either text/plain or application/json>",
  "pred_x_csv": "<The comma-separated x column values to be used for prediction>"
}`

This script will be packaged into the container that will be built in the upcoming steps.

You can view the script by running the following code cell.

In [ ]:
# View the Python3 Flask script (containing the inference code)
!cat {container_script_file}

### D) Create the Dockerfile <a id='Create%20the%20Dockerfile'></a>

In this step, we will create a [Dockerfile](https://docs.docker.com/engine/reference/builder/) which is required to build our [Docker](https://www.docker.com/) container containing the model pickle file, an inference script and its dependencies.

The container uses the [Amazon Linux 2023 container image](https://gallery.ecr.aws/amazonlinux/amazonlinux) from the [Amazon ECR public registry](https://aws.amazon.com/ecr/) as the base image.  As this is a public registry, you do not require any credentials or permissions to download it.

In [ ]:
# Copy the inference script and requirements.txt to the container-artifacts directory
!cp -pr {container_script_file} {container_artifacts_dir}/server.py
!cp -pr {container_script_req_file} {container_artifacts_dir}/requirements.txt

# Create the Dockerfile content
# Note: This uses Amazon Linux 2023 as the base image for the *container* (not the host OS).
# The container image is pulled from the ECR public registry and does not depend on the host OS.
dockerfile_content = f"""# Use Amazon Linux 2023 as the base image
FROM public.ecr.aws/amazonlinux/amazonlinux:latest

# Setup the working directory
WORKDIR /

# Install Python3 and pip (dnf is the package manager in AL2023)
RUN dnf -y install python3 python3-pip && dnf clean all

# Setup the Python virtual env to run the inference script
RUN python3 -m venv /opt/appenv

# Install the Python packages required for the inference script in the virtual env
COPY requirements.txt .
RUN /opt/appenv/bin/pip install --no-cache-dir -r requirements.txt

# Copy the extracted model file and the inference script
COPY {extracted_model_file_name} ./
COPY server.py ./

# Specify the ENV variables
ENV MODEL_PICKLE_FILE_PATH={extracted_model_file_name}
ENV FLASK_SERVER_LOG_LEVEL=DEBUG
ENV FLASK_SERVER_HOSTNAME=0.0.0.0
ENV FLASK_SERVER_PORT={ecs_container_port}
ENV FLASK_SERVER_DEBUG=True

# Specify the command to run the inference script as a Flask app
ENTRYPOINT ["/opt/appenv/bin/python", "server.py"]
"""

# Create the Dockerfile
dockerfile_local_path = f'{container_artifacts_dir}/Dockerfile'
with open(dockerfile_local_path, 'wt') as file:
    file.write(dockerfile_content)

# Print the contents of the generated Dockerfile
!cat {dockerfile_local_path}

### E) Create the container <a id='Create%20the%20container'></a>

Create the Docker container using the `docker build` command.  Specify the container image name and point to the container-artifacts directory that contains all the files to build the container.

Note: You may see warning messages when the container is built.  These warnings will be around installing the Python packages that are required by the inference script.  You can choose to either ignore or fix them.

In [ ]:
# Build the Docker container
!docker build -t {container_image_name} {container_artifacts_dir}

### F) Create the private repository in ECR <a id='Create%20the%20private%20repository%20in%20ECR'></a>

In order to configure Amazon ECS to run a container, the container image should exist in a container registry.  In this notebook, we will create and use an [Amazon ECR](https://aws.amazon.com/ecr/) private repository for this purpose.

In this step, we will check if the private repository in Amazon ECR that we intend to create already exists or not.  If it does not exist, we will create it with the repository name the same as the container image name.

Note: When creating the repository, setting the `scanOnPush` parameter to `True` will automatically initiate a vulnerability scan on the container image that is pushed to the repository.  For more info on image scanning, refer [here](https://docs.aws.amazon.com/AmazonECR/latest/userguide/image-scanning.html).

In [ ]:
# Check if the ECR repository exists already; if not, then create it
try:
    ecr_client.describe_repositories(repositoryNames=[container_image_name])
    print('ECR repository {} already exists.'.format(container_image_name))
except ecr_client.exceptions.RepositoryNotFoundException:
    print('ECR repository {} does not exist.'.format(container_image_name))
    print('Creating ECR repository {}...'.format(container_image_name))
    # Create the ECR repository - here we use the container image name for the repository name
    ecr_client.create_repository(repositoryName=container_image_name,
                                 imageScanningConfiguration={
                                     'scanOnPush': True
                                 })
    print('Completed creating ECR repository {}.'.format(container_image_name))

### G) Push the container to ECR <a id='Push%20the%20container%20to%20ECR'></a>

In this step, we will tag and push the container to the private ECR registry created in step F.

The cell below authenticates Docker to ECR with an explicit `docker login` using a short-lived token from `aws ecr get-login-password`.  This works reliably even if the ECR credential helper configured in the prerequisites step was not picked up, which is a common cause of `no basic auth credentials` errors during `docker push`.  The commands run through `subprocess` so that any failure raises a Python error and stops the cell (a bare `!docker push` would fail silently).

In [ ]:
import subprocess

# Set the image names
source_image_name = f'{container_image_name}:latest'
target_image_name = f'{container_registry_url_prefix}/{container_image_name}:latest'


def run(cmd, **kwargs):
    """Run a command, streaming output, and raise if it fails."""
    print(f'$ {" ".join(cmd)}')
    subprocess.run(cmd, check=True, **kwargs)


# Authenticate Docker to the ECR registry with a short-lived token.
# This is the most reliable method and does not depend on the credential helper.
login_password = subprocess.check_output(
    ['aws', 'ecr', 'get-login-password', '--region', region_name],
    text=True
).strip()
run(
    ['docker', 'login', '--username', 'AWS', '--password-stdin', container_registry_url_prefix],
    input=login_password, text=True
)

# Tag the container
run(['docker', 'tag', source_image_name, target_image_name])

# Push the container to the specified registry in Amazon ECR.
# check=True ensures a failed push raises CalledProcessError instead of failing silently.
run(['docker', 'push', target_image_name])

print(f'\nPushed image: {target_image_name}')

##  5. Deploy and test on Amazon ECS on AWS Fargate <a id='Deploy%20and%20test%20on%20Amazon%20ECS%20on%20AWS%20Fargate'></a>

In this step, we will create an [Amazon ECS cluster](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/clusters.html), deploy the Docker container that was created in the previous step as a [task](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/task_definitions.html) on [Amazon ECS on AWS Fargate](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/AWS_Fargate.html) and test it.

### A) Create the ECS cluster <a id='Create%20the%20ECS%20cluster'></a>

In this step, we will check if the ECS cluster that we intend to create already exists or not.  If it does not exist, we will create it.

Note:

* We have not configured this cluster to use an [Amazon VPC](https://aws.amazon.com/vpc) for networking.  If you require it, refer to the instructions [here](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/create_cluster.html).
* Sometimes, after the `delete_cluster` API is invoked on an ECS cluster, the cluster can go into 'INACTIVE' state and may remain discoverable in your AWS account for a period of time.  You may not see this in the AWS console.  When this happens, you won't be able to use the cluster.  For more information on this, refer [here](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/ecs.html#ECS.Client.delete_cluster).

In [ ]:
# Check if the Amazon ECS cluster exists already; if not, then create it.
# describe_clusters returns matches under 'clusters' and unmatched names under
# 'failures' (with reason 'MISSING'). A cluster that was previously deleted can
# linger in 'INACTIVE' state and cannot be reused, so we recreate it in that case.
describe_ecs_cluster_response = ecs_client.describe_clusters(clusters=[ecs_cluster_name])
clusters = describe_ecs_cluster_response['clusters']

existing_status = clusters[0]['status'] if clusters else None

if existing_status == 'ACTIVE':
    print("ECS cluster '{}' already exists and is ACTIVE.".format(ecs_cluster_name))
else:
    if existing_status == 'INACTIVE':
        print("ECS cluster '{}' exists but is INACTIVE; recreating it.".format(ecs_cluster_name))
    else:
        print('ECS cluster {} does not exist.'.format(ecs_cluster_name))
    print('Creating ECS cluster {}...'.format(ecs_cluster_name))
    create_ecs_cluster_response = ecs_client.create_cluster(clusterName=ecs_cluster_name)
    print('ECS cluster status = {}'.format(create_ecs_cluster_response['cluster']['status']))

In [ ]:
# Sleep every 10 seconds and print the status of the ECS cluster until it goes to ACTIVE, INACTIVE or FAILED state
while True:
    describe_ecs_cluster_response = ecs_client.describe_clusters(clusters=[ecs_cluster_name])
    ecs_cluster_status = describe_ecs_cluster_response['clusters'][0]['status']
    print('ECS cluster status = {}'.format(ecs_cluster_status))
    if ecs_cluster_status in {'ACTIVE', 'INACTIVE', 'FAILED'}:
        break
    time.sleep(10)

### B) Create the ECS Task and deploy the container <a id='Create%20the%20ECS%20Task%20and%20deploy%20the%20container'></a>

In this step, we will create a [task](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/task_definitions.html) on [Amazon ECS on AWS Fargate](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/AWS_Fargate.html) and deploy the container.

The following configuration will be used:

* Fargate launch type.  For details on how to configure this, refer [here](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/AWS_Fargate.html).
* The container location will be the Amazon ECR private registry that we created in prior steps.
* The container port and host port will be what we configured in prior steps.
* Protocol will be TCP.
* `awslogs` driver will be used to send the logs to [Amazon CloudWatch](https://aws.amazon.com/cloudwatch/).
* Healthcheck will be configured to invoke the healthcheck URL path suffix configured in the inference script's Flask app running in the container.
* CPU and memory settings will be what we configured in prior steps.
* The cluster name, container count, VPC subnets and Security Groups will be what we configured in prior steps.
* Auto-assign Public IP address to the Task.

In [ ]:
# Register the ECS Fargate task definition
ecs_register_task_definition_response = ecs_client.register_task_definition(family=ecs_fargate_task_name,
                                                                            taskRoleArn=ecs_fargate_task_role,
                                                                            executionRoleArn=ecs_fargate_task_execution_role,
                                                                            networkMode='awsvpc',
                                                                            containerDefinitions=[{
                                                                                'name':ecs_container_name,
                                                                                'image':target_image_name,
                                                                                'portMappings': [{
                                                                                    'containerPort':ecs_container_port,
                                                                                    'hostPort':ecs_container_host_port,
                                                                                    'protocol':'tcp',
                                                                                }],
                                                                                'logConfiguration': {
                                                                                    'logDriver':'awslogs',
                                                                                    'options': {
                                                                                        'awslogs-create-group':'true',
                                                                                        'awslogs-region':region_name,
                                                                                        'awslogs-group':'/ecs/{}'.format(ecs_fargate_task_name),
                                                                                        'awslogs-stream-prefix':'ecs'
                                                                                    }
                                                                                },
                                                                                'healthCheck': {
                                                                                    'command':ecs_container_healthcheck_command_list,
                                                                                    'interval':ecs_container_healthcheck_interval_in_seconds,
                                                                                    'timeout':ecs_container_healthcheck_timeout_in_seconds
                                                                                }
                                                                            }],
                                                                            requiresCompatibilities=[
                                                                                'FARGATE'
                                                                            ],
                                                                            cpu=ecs_fargate_task_cpu,
                                                                            memory=ecs_fargate_task_memory
                                                                           )


# Print the task definition ARN
ecs_fargate_task_definiton_arn = ecs_register_task_definition_response['taskDefinition']['taskDefinitionArn']
print('ECS Fargate Task definition ARN = {}'.format(ecs_fargate_task_definiton_arn))

The ECS cluster created in the previous step will take a few seconds to go into ACTIVE state.  Wait until then and proceed to the next step.

In [ ]:
# Run the ECS Fargate task
ecs_run_task_response = ecs_client.run_task(cluster=ecs_cluster_name,
                                            count=ecs_fargate_task_count,
                                            launchType='FARGATE',
                                            networkConfiguration={
                                                'awsvpcConfiguration':{
                                                    'subnets':ecs_fargate_task_subnet_list,
                                                    'securityGroups':ecs_fargate_task_security_group_list,
                                                    'assignPublicIp': 'ENABLED'
                                                }
                                            },
                                            taskDefinition=ecs_fargate_task_name)

# run_task can return an empty 'tasks' list with details under 'failures'
# (e.g., invalid subnet/security group, or capacity issues). Surface these
# instead of raising an opaque IndexError on the next line.
if ecs_run_task_response.get('failures'):
    raise RuntimeError('run_task failed: {}'.format(ecs_run_task_response['failures']))

# Print the task ARN
ecs_fargate_task_id = ecs_run_task_response['tasks'][0]['taskArn']
print('ECS Fargate Task ARN = {}'.format(ecs_fargate_task_id))

### C) Prepare to test the ECS Task <a id='Prepare%20to%20test%20the%20ECS%20Task'></a>

Follow these steps to prepare to the test the ECS Task:

1. **Wait for the Fargate task state to go to `RUNNING`** - after the successful run of the previous step, it will take few minutes for the ECS Fargate Task to go into `RUNNING` state.  You can check the state either using the AWS CLI/API/SDK or by going into the [AWS console](https://console.aws.amazon.com/ecs/home).  In the ECS console page for the AWS region, navigate to the specific ECS cluster that was created by this notebook and go to the Tasks tab.  There, you can check the state of your ECS Fargate Task.

In [ ]:
# Sleep every 5 seconds and print the status of the ECS Task until it goes to RUNNING or STOPPED state
while True:
    ecs_describe_tasks_response = ecs_client.describe_tasks(cluster=ecs_cluster_name,
                                                            tasks=[ecs_fargate_task_id])
    ecs_task = ecs_describe_tasks_response['tasks'][0]
    ecs_task_status = ecs_task['lastStatus']
    print('ECS Fargate Task status = {}'.format(ecs_task_status))
    if ecs_task_status in {'RUNNING', 'STOPPED'}:
        break
    time.sleep(5)

# If the task stopped instead of running, surface the reason so the failure is
# actionable (common causes: image pull failure, failed health check, or the
# task role/execution role missing permissions).
if ecs_task_status == 'STOPPED':
    print('\nTask STOPPED. Reason: {}'.format(ecs_task.get('stoppedReason', 'unknown')))
    for container in ecs_task.get('containers', []):
        print('  Container \'{}\': {} (exit code: {}, reason: {})'.format(
            container.get('name'),
            container.get('lastStatus'),
            container.get('exitCode', 'n/a'),
            container.get('reason', 'n/a')))
    print('\nTip: check the container logs in CloudWatch Logs group '
          '\'/ecs/{}\' for details.'.format(ecs_fargate_task_name))

2. **Retrieve the Public IP address of this environment** - if you intend to test the ECS Task from this notebook, then you will require the Public IP address of this environment to configure in the next step.  You can retrieve it by running the following code cell.

In [ ]:
# Print the Public IP address of this environment
!curl -s ifconfig.me

3. **Configure the Security Group on the ECS Task** - setup an Inbound Rule in the ECS Task's Security Group to allow access to the IP address of the system from where you are going to test the ECS Task.  This rule should be for the HTTP protocol on the configured port.  In this notebook, we have configured a Public IP address on the ECS Task on port 80.  We will test the ECS Task from this notebook.  So, make sure you configure the environment's Public IP address retrieved in the previous step in this Security Group.

### D) Test the ECS Task <a id='Test%20the%20ECS%20Task'></a>

In this step, we will test the ECS Task that we created in the previous step by invoking it synchronously.  For this, we will invoke the Python3 [Flask](https://flask.palletsprojects.com/en/1.1.x/) app script running in the container by using the Public IP address of the ECS Task.

1. Retrieve the Public IP address of the ECS Task.
2. Invoke the endpoint by making a HTTP POST call with the first record of the test dataset as a CSV string.
    The request should be in the following format:
    `{
      "response_content_type": "<Specify either text/plain or application/json>",
      "pred_x_csv": "<The comma-separated x column values to be used for prediction>"
    }`

In [ ]:
# Retrieve the ECS Task details
ecs_describe_tasks_response = ecs_client.describe_tasks(cluster=ecs_cluster_name,
                                                       tasks=[ecs_fargate_task_id])

# Retrieve the Public IP address of the ECS Task
ecs_task_attachments = ecs_describe_tasks_response['tasks'][0]['attachments']
for ecs_task_attachment in ecs_task_attachments:
    if ecs_task_attachment['type'] == 'ElasticNetworkInterface':
        ecs_task_attachment_details = ecs_task_attachment['details']
        for ecs_task_attachment_detail in ecs_task_attachment_details:
            if ecs_task_attachment_detail['name'] == 'networkInterfaceId':
                ecs_task_nid = ecs_task_attachment_detail['value']
                describe_network_interfaces_response = ec2_client.describe_network_interfaces(NetworkInterfaceIds=[
                    ecs_task_nid
                ])
                ecs_fargate_task_public_ip = describe_network_interfaces_response['NetworkInterfaces'][0]['Association']['PublicIp']
                
# Print the Public IP address of the ECS Task
print('ECS Task Public IP address = {}'.format(ecs_fargate_task_public_ip))

In [ ]:
# Set the request payload
x_test_request_payload_csv = ','.join(map(str, x_test[0]))
x_test_request_payload = '{' + '"response_content_type": "application/json","pred_x_csv":"{}"'.format(x_test_request_payload_csv) + '}'
# Print the request
print('Request payload:\n')
print(x_test_request_payload)

# Invoke the ECS Task and print the response
ecs_fargate_task_public_url = 'http://{}:80/'.format(ecs_fargate_task_public_ip)
print('\nResponse:\n')
!curl -X POST -H 'Content-Type: application/json' --data '{x_test_request_payload}' {ecs_fargate_task_public_url}

##  6. (Optional) Front-end the container with Amazon API Gateway <a id='(Optional)%20Front-end%20the%20container%20with%20Amazon%20API%20Gateway'></a>

For some use cases, you may prefer to front-end the inference as a service on Amazon ECS on AWS Fargate with [Amazon API Gateway](https://docs.aws.amazon.com/apigateway/latest/developerguide/welcome.html).  With this setup, you can serve the model inference as an API with a HTTPS endpoint.  Prior to setting up the API, you have to create an [Amazon ECS Service](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/ecs_services.html) made up of multiple tasks and then setup [Load Balancing](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/service-load-balancing.html) and [Auto Scaling](https://docs.aws.amazon.com/AmazonECS/latest/developerguide/service-auto-scaling.html).

For the API, you have the following options to choose from:
* [HTTP API](https://docs.aws.amazon.com/apigateway/latest/developerguide/http-api.html)
* [REST API](https://docs.aws.amazon.com/apigateway/latest/developerguide/apigateway-rest-api.html)

For guidance on choosing the right API option, refere [here](https://docs.aws.amazon.com/apigateway/latest/developerguide/http-api-vs-rest.html).

For information on setting up the container as the backend for Amazon API Gateway, refer [here](https://docs.aws.amazon.com/apigateway/latest/developerguide/setup-http-integrations.html).

Note: The container that we created in prior steps has the logic to handle both REST and HTTP API requests from the Amazon API Gateway assuming the gateway passes through the request payload as-is to the backend container.

## 7. Cleanup <a id='Cleanup'></a>

As a best practice, you should delete resources and S3 objects when no longer required.  This will help you avoid incurring unnecessary costs.

This step will cleanup the resources and S3 objects created by this notebook.

Note: Apart from these resources, there will be Docker containers and related images created in this environment.  If you decide to delete them, open a Terminal and run appropriate `docker` commands (e.g., `docker system prune`).

### A) Cleanup ECS resources <a id='Cleanup%20ECS%20resources'></a>

Note: Sometimes, after the `delete_cluster` API is invoked on an ECS cluster, the cluster can go into 'INACTIVE' state and may remain discoverable in your AWS account for a period of time.  You may not see this in the AWS console.  When this happens, you won't be able to use the cluster.  For more information on this, refer [here](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/ecs.html#ECS.Client.delete_cluster).

In [ ]:
# Stop the ECS Task
ecs_client.stop_task(cluster=ecs_cluster_name,
                     task=ecs_fargate_task_id,
                     reason='Cleanup from notebook {}'.format(nb_name))

In [ ]:
# Deregister ECS Task definition
ecs_client.deregister_task_definition(taskDefinition=ecs_fargate_task_definiton_arn)

In [ ]:
# Delete the ECS cluster
ecs_client.delete_cluster(cluster=ecs_cluster_name)

### B) Cleanup ECR repository <a id='Cleanup%20ECR%20repository'></a>

In [ ]:
# Delete the ECR private repository
try:
    ecr_client.delete_repository(repositoryName=container_image_name, force=True)
    print('ECR repository {} deleted.'.format(container_image_name))
except ecr_client.exceptions.RepositoryNotFoundException:
    print('ECR repository {} does not exist.'.format(container_image_name))

### C) Cleanup S3 objects <a id='Cleanup%20S3%20objects'></a>

In [ ]:
# Delete data from S3 bucket
for file in s3_bucket_resource.objects.filter(Prefix='{}/'.format(nb_name)):
    file_key = file.key
    print('Deleting {} ...'.format(file_key))
    s3_resource.Object(s3_bucket_resource.name, file_key).delete()